In [1]:
# takes in two strings and returns optimal local alignment and score
# asked ChatGPT how to make a matrix of 0s
# https://www.delftstack.com/howto/python/smith-waterman-algorithm-python/

seq1 = 'ACTACGCA'
seq2 = 'TATGC'
match_score = 1       # score for a match
mismatch_penalty = -1 # penalty for mismatch
gap_penalty = -1      # penalty for a gap

rows = len(seq2) + 1 #matrix size of sequences
cols = len(seq1) + 1 
matrix = [[0 for _ in range(cols)] for _ in range(rows)] # Create 2D matrix filled with zeros

# Initialize first row and first column with gap penalties
for i in range(rows):
    matrix[i][0] = i * gap_penalty
for j in range(cols):
    matrix[0][j] = j * gap_penalty

for i in range(1, rows):
    for j in range(1, cols):
        if seq1[j-1] == seq2[i-1]:
            diag = matrix[i-1][j-1] + match_score
        else:
            diag = matrix[i-1][j-1] + mismatch_penalty
        
        up = matrix[i-1][j] + gap_penalty
        left = matrix[i][j-1] + gap_penalty
        
        matrix[i][j] = max(diag, up, left)

aligned_seq1 = ""
aligned_seq2 = ""

i = rows - 1
j = cols - 1

while i > 0 and j > 0:
    score_current = matrix[i][j]
    score_diag = matrix[i-1][j-1]
    score_up = matrix[i-1][j]
    score_left = matrix[i][j-1]

    if seq1[j-1] == seq2[i-1]:
        match = match_score
    else:
        match = mismatch_penalty

    # Check if coming from diagonal
    if score_current == score_diag + match:
        aligned_seq1 = seq1[j-1] + aligned_seq1
        aligned_seq2 = seq2[i-1] + aligned_seq2
        i -= 1
        j -= 1
    # Check if coming from left (gap in seq2)
    elif score_current == score_left + gap_penalty:
        aligned_seq1 = seq1[j-1] + aligned_seq1
        aligned_seq2 = "-" + aligned_seq2
        j -= 1
    # Check if coming from up (gap in seq1)
    else:
        aligned_seq1 = "-" + aligned_seq1
        aligned_seq2 = seq2[i-1] + aligned_seq2
        i -= 1

# Fill remaining gaps if any sequence is not fully aligned
while j > 0:
    aligned_seq1 = seq1[j-1] + aligned_seq1
    aligned_seq2 = "-" + aligned_seq2
    j -= 1

while i > 0:
    aligned_seq1 = "-" + aligned_seq1
    aligned_seq2 = seq2[i-1] + aligned_seq2
    i -= 1

for row in matrix:
    print(row)



[0, -1, -2, -3, -4, -5, -6, -7, -8]
[-1, -1, -2, -1, -2, -3, -4, -5, -6]
[-2, 0, -1, -2, 0, -1, -2, -3, -4]
[-3, -1, -1, 0, -1, -1, -2, -3, -4]
[-4, -2, -2, -1, -1, -2, 0, -1, -2]
[-5, -3, -1, -2, -2, 0, -1, 1, 0]


In [6]:
def smith_waterman(seq1, seq2, match_score=1, mismatch_penalty=1, gap_penalty=1): 
    max_score = 0
    max_pos = (0,0)
    
    rows = len(seq2) + 1 #matrix size 
    cols = len(seq1) + 1 
    matrix = [[0 for _ in range(cols)] for _ in range(rows)] # Create scoring matrix filled with zeros
       
    # Score
    for i in range(1, rows): # first row/column is 0 so starts in second
        for j in range(1, cols):
            if seq1[j-1] == seq2[i-1]:
                diag = matrix[i-1][j-1] + match_score # if two align get match point
            else:
                diag = matrix[i-1][j-1] - mismatch_penalty # if two do not align, mismatch penalty

            up = matrix[i-1][j] - gap_penalty # puts gap penalty in seq1
            left = matrix[i][j-1] - gap_penalty # puts gap penalty in seq2

            matrix[i][j] = max(0, diag, up, left) # highest score 
            if matrix[i][j] > max_score:
                max_score = matrix[i][j] # keeps track of highest score
                max_pos = (i, j) # where highest score is 
    # now the matrix is scored 
    # reconstructs which two aligned make best match
    aligned_seq1 = ""
    aligned_seq2 = ""
    i, j = max_pos # start from cell with highest score 

    while matrix[i][j] != 0: # trace backwards until reach a 0 ie a stop does not match
        score_current = matrix[i][j] # score of current
        score_diag = matrix[i-1][j-1] # score of diagonal
        score_up = matrix[i-1][j] # score of above
        score_left = matrix[i][j-1] # score of left - which could have to this one

        if seq1[j-1] == seq2[i-1]:
            match = match_score
        else:
            match = mismatch_penalty # is diagonal a match or mismatch 
        # how did we arrive at the cell - diagonal, left (gap in seq2), or up (gap in seq1)
        if score_current == score_diag + match:
            aligned_seq1 = seq1[j-1] + aligned_seq1
            aligned_seq2 = seq2[i-1] + aligned_seq2
            i -= 1
            j -= 1 # match was diagnoal; add to strink and move diagonally up and left 
        elif score_current == score_left + gap_penalty:
            aligned_seq1 = seq1[j-1] + aligned_seq1
            aligned_seq2 = "-" + aligned_seq2
            j -= 1 # came from left meaning seq2 has a gap, add - in a seq2 and move left 
        else:
            aligned_seq1 = "-" + aligned_seq1
            aligned_seq2 = seq2[i-1] + aligned_seq2
            i -= 1 # came from top meaning seq1 has a gap, add - in seq1 and move up

    # Print full matrix
    for row in matrix:
        print(row) 

    return aligned_seq1, aligned_seq2, max_score # return the aligned sequence and max score 

In [7]:
# test alignment of sequences

smith_waterman('TACG', 'TATG')

[0, 0, 0, 0, 0]
[0, 1, 0, 0, 0]
[0, 0, 2, 1, 0]
[0, 1, 1, 1, 0]
[0, 0, 0, 0, 2]


('TA', 'TA', 2)

In [8]:
# test using examples in problem set

smith_waterman('tgcatcgagaccctacgtgac', 'actagacctagcatcgac')

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0]
[0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 2, 1, 1, 0, 0, 2, 1, 0, 0, 0, 2]
[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 2, 1, 1, 1, 2, 1, 0, 1]
[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 3, 2, 1, 1, 1, 2, 1]
[0, 0, 1, 0, 0, 0, 0, 1, 0, 2, 1, 0, 0, 0, 0, 2, 2, 3, 2, 2, 1, 1]
[0, 0, 0, 0, 1, 0, 0, 0, 2, 1, 3, 2, 1, 0, 0, 1, 1, 2, 2, 1, 3, 2]
[0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 2, 4, 3, 2, 1, 0, 2, 1, 1, 1, 2, 4]
[0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 3, 5, 4, 3, 2, 1, 1, 0, 0, 1, 3]
[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 4, 4, 5, 4, 3, 2, 2, 1, 0, 2]
[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 3, 3, 4, 6, 5, 4, 3, 2, 2, 1]
[0, 0, 1, 0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 3, 5, 5, 6, 5, 4, 3, 2]
[0, 0, 0, 2, 1, 0, 1, 0, 0, 1, 1, 2, 1, 3, 2, 4, 6, 5, 5, 4, 3, 4]
[0, 0, 0, 1, 3, 2, 1, 0, 1, 0, 2, 1, 1, 2, 2, 3, 5, 5, 4, 4, 5, 4]
[0, 1, 0, 0, 2, 4, 3, 2, 1, 0, 1, 1, 0, 1, 3, 2, 4, 4, 6, 5, 4

('cctacg---tgac', 'acctagcatcgac', 8)

In [9]:
# test using examples in problem set

smith_waterman('tgcatcgagaccctacgtgac', 'actagacctagcatcgac', gap_penalty=2)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0]
[0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 2, 1, 1, 0, 0, 2, 0, 0, 0, 0, 2]
[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 2, 0, 0, 1, 1, 0, 0, 0]
[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 3, 1, 0, 0, 0, 1, 0]
[0, 0, 1, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 0, 1, 2, 2, 0, 1, 0, 0]
[0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 3, 1, 0, 0, 0, 1, 0, 1, 1, 0, 2, 0]
[0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 4, 2, 1, 0, 0, 2, 0, 0, 0, 0, 3]
[0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 2, 5, 3, 1, 0, 1, 1, 0, 0, 0, 1]
[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 3, 4, 4, 2, 0, 0, 2, 0, 0, 0]
[0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 2, 3, 5, 3, 1, 0, 1, 1, 0]
[0, 0, 1, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 0, 1, 3, 4, 4, 2, 1, 0, 0]
[0, 0, 0, 2, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 4, 3, 3, 1, 0, 1]
[0, 0, 0, 0, 3, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 2, 3, 2, 2, 2, 0]
[0, 1, 0, 0, 1, 4, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 4, 2, 1

('gcatcga', 'gcatcga', 7)